# Day 12 — Evaluation suite

Batch evaluation for refusal rate, confidence, citation coverage.

In [ ]:
!pip -q install pandas numpy faiss-cpu

In [ ]:
# === Setup ===
from google.colab import drive
drive.mount('/content/drive')

import os, re, time, uuid
import numpy as np
import pandas as pd
from tqdm import tqdm

BASE = "/content/drive/MyDrive/mimic_project"
print("BASE:", BASE)


In [ ]:
import faiss
df = pd.read_csv(f"{BASE}/clean_dataset_2696.csv")
fused = np.load(f"{BASE}/fused_embeddings_alpha_0.5.npy").astype("float32")
fused = fused / np.linalg.norm(fused, axis=1, keepdims=True)
index = faiss.IndexFlatIP(fused.shape[1]); index.add(fused)

def retrieve_cases(qi, k=3):
    s, idx = index.search(fused[qi:qi+1], k+1)
    idxs = [j for j in idx[0].tolist() if j != qi][:k]
    cases=[]
    for ci,j in enumerate(idxs, start=1):
        cases.append({"case":ci,"study_id":int(df.loc[j,"study_id"]),"impression":str(df.loc[j,"impression"])})
    best = float(s[0][1]) if len(s[0])>1 else 0.0
    return cases, best

def draft(cases):
    bullets=[]
    for c in cases:
        txt=c["impression"].replace("\n"," ").strip()
        first=re.split(r"(?<=[.!?])\s+", txt)[0].strip()
        bullets.append(f"- {first} [Case {c['case']}]")
    return "IMPRESSION:\n" + "\n".join(bullets)

def citation_cov(d,k=3):
    needed=[f"[Case {i}]" for i in range(1,k+1)]
    return sum(1 for x in needed if x in d)/len(needed)

def eval_rag(n=500,k=3,min_score=0.90):
    rows=[]
    for qi in tqdm(range(min(n,len(df)))):
        cases,best=retrieve_cases(qi,k=k)
        d=draft(cases)
        cov=citation_cov(d,k=k)
        status="generated" if best>=min_score else "refused"
        rows.append({"query_study_id":int(df.loc[qi,"study_id"]), "best_score":best,
                     "status":status, "citation_coverage":cov})
    return pd.DataFrame(rows)

out = eval_rag()
out_path = f"{BASE}/eval_day12_rag.csv"
out.to_csv(out_path,index=False)
print("Saved:", out_path)
print("Refusal rate:", float((out.status=='refused').mean()))
print("Avg best_score:", float(out.best_score.mean()))
print("Avg citation coverage:", float(out.citation_coverage.mean()))
